In [1]:
import sys
print(f"Python version: {sys.version}")

import autogen_core
print(f"AutoGen version: {autogen_core.__version__}")

print("\n[SUCCESS] All packages verified!")

Python version: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
AutoGen version: 0.7.5

[SUCCESS] All packages verified!


In [2]:
from dataclasses import dataclass
from autogen_core import (
    MessageContext,
    RoutedAgent,
    message_handler,
    AgentId,
    SingleThreadedAgentRuntime
)
import asyncio
from datetime import datetime

print("✅ All imports successful!")

✅ All imports successful!


In [3]:
@dataclass
class Question:
    """Student's question to teacher"""
    topic: str
    question_text: str
    student_name: str
    timestamp: str = None
    
    def __post_init__(self):
        if self.timestamp is None:
            self.timestamp = datetime.now().strftime('%H:%M:%S')


@dataclass
class Answer:
    """Teacher's answer to student"""
    original_question: str
    answer_text: str
    additional_resources: str
    teacher_name: str
    timestamp: str = None
    
    def __post_init__(self):
        if self.timestamp is None:
            self.timestamp = datetime.now().strftime('%H:%M:%S')


@dataclass
class ThankYou:
    """Student's acknowledgment"""
    message: str
    learned_topic: str
    timestamp: str = None
    
    def __post_init__(self):
        if self.timestamp is None:
            self.timestamp = datetime.now().strftime('%H:%M:%S')

print("✅ Message types defined!")

✅ Message types defined!


In [4]:
class StudentAgent(RoutedAgent):
    """Agent that asks questions and learns"""
    
    def __init__(self, name: str) -> None:
        super().__init__(name)
        self._name = name
        self._questions_asked = 0
        self._topics_learned = []
        print(f"[INIT] {self._name} is ready to learn!")
    
    @message_handler
    async def handle_answer(self, message: Answer, ctx: MessageContext) -> None:
        """Receive and process teacher's answer"""
        print(f"\n{'='*70}")
        print(f"[{self._name}] 📚 Received answer from {message.teacher_name}")
        print(f"{'='*70}")
        print(f"Question: {message.original_question}")
        print(f"Answer: {message.answer_text}")
        print(f"Resources: {message.additional_resources}")
        print(f"{'='*70}\n")
        
        # Store learned topic
        topic = message.original_question.split()[0]
        self._topics_learned.append(topic)
        
        # Send thank you message
        thank_you = ThankYou(
            message=f"Thank you for explaining! I understand {topic} better now.",
            learned_topic=topic
        )
        
        print(f"[{self._name}] 🙏 Sending: '{thank_you.message}'")
        await self.send_message(thank_you, AgentId("teacher_agent", "default"))
    
    def get_stats(self):
        """Get learning statistics"""
        return {
            'questions_asked': self._questions_asked,
            'topics_learned': self._topics_learned
        }

print("✅ StudentAgent defined!")


✅ StudentAgent defined!


In [5]:
class TeacherAgent(RoutedAgent):
    """Agent that answers questions and provides guidance"""
    
    def __init__(self, name: str) -> None:
        super().__init__(name)
        self._name = name
        self._questions_answered = 0
        self._knowledge_base = {
            'Python': {
                'answer': 'Python is a high-level programming language known for its simplicity and readability. It uses indentation for code blocks and supports multiple programming paradigms.',
                'resources': 'Check out python.org, Real Python, and Python documentation'
            },
            'Agents': {
                'answer': 'Agents are autonomous software entities that can perceive their environment, make decisions, and take actions to achieve specific goals. They communicate via messages.',
                'resources': 'Study Microsoft AutoGen, LangChain agents, and multi-agent systems'
            },
            'AI': {
                'answer': 'Artificial Intelligence (AI) refers to computer systems that can perform tasks typically requiring human intelligence, such as learning, reasoning, and problem-solving.',
                'resources': 'Explore machine learning courses, AI papers, and frameworks like TensorFlow'
            }
        }
        print(f"[INIT] {self._name} is ready to teach!")
    
    @message_handler
    async def handle_question(self, message: Question, ctx: MessageContext) -> None:
        """Receive and answer student's question"""
        print(f"\n{'='*70}")
        print(f"[{self._name}] 📝 Received question from {message.student_name}")
        print(f"{'='*70}")
        print(f"Topic: {message.topic}")
        print(f"Question: {message.question_text}")
        print(f"{'='*70}\n")
        
        self._questions_answered += 1
        
        # Get answer from knowledge base
        knowledge = self._knowledge_base.get(
            message.topic,
            {
                'answer': f"That's an interesting question about {message.topic}! Let me provide a general overview.",
                'resources': 'Search online resources and academic papers'
            }
        )
        
        print(f"[{self._name}] 🧠 Preparing answer...")
        
        # Create answer
        answer = Answer(
            original_question=message.question_text,
            answer_text=knowledge['answer'],
            additional_resources=knowledge['resources'],
            teacher_name=self._name
        )
        
        print(f"[{self._name}] 📤 Sending answer back to student...\n")
        await self.send_message(answer, AgentId("student_agent", "default"))
    
    @message_handler
    async def handle_thank_you(self, message: ThankYou, ctx: MessageContext) -> None:
        """Receive student's acknowledgment"""
        print(f"\n[{self._name}] 😊 Received: '{message.message}'")
        print(f"[{self._name}] Great! Happy to help you learn about {message.learned_topic}!\n")
    
    def get_stats(self):
        """Get teaching statistics"""
        return {
            'questions_answered': self._questions_answered,
            'topics_available': list(self._knowledge_base.keys())
        }

print("✅ TeacherAgent defined!")

✅ TeacherAgent defined!


In [6]:
print("="*70)
print(" "*15 + "TEACHER-STUDENT AGENT SYSTEM")
print("="*70)

# Create runtime
print("\n[SYSTEM] 🚀 Initializing learning environment...")
runtime = SingleThreadedAgentRuntime()

# Register agents
print("[SYSTEM] 📋 Registering agents...\n")

await StudentAgent.register(
    runtime,
    "student_agent",
    lambda: StudentAgent("Sai")
)

await TeacherAgent.register(
    runtime,
    "teacher_agent",
    lambda: TeacherAgent("Professor Smith")
)

print("\n[SYSTEM] ✅ Both agents registered!")

# Start runtime
print("[SYSTEM] ▶️  Starting runtime...\n")
runtime.start()

print("✅ Runtime is active and ready!")
print("\n" + "="*70)
print(" "*20 + "LEARNING SESSION BEGINS")
print("="*70)

               TEACHER-STUDENT AGENT SYSTEM

[SYSTEM] 🚀 Initializing learning environment...
[SYSTEM] 📋 Registering agents...


[SYSTEM] ✅ Both agents registered!
[SYSTEM] ▶️  Starting runtime...

✅ Runtime is active and ready!

                    LEARNING SESSION BEGINS


[INIT] Professor Smith is ready to teach!

[Professor Smith] 📝 Received question from Sai
Topic: Python
Question: Python - What is Python and why is it popular?

[Professor Smith] 🧠 Preparing answer...
[Professor Smith] 📤 Sending answer back to student...

[INIT] Sai is ready to learn!

[Sai] 📚 Received answer from Professor Smith
Question: Python - What is Python and why is it popular?
Answer: Python is a high-level programming language known for its simplicity and readability. It uses indentation for code blocks and supports multiple programming paradigms.
Resources: Check out python.org, Real Python, and Python documentation

[Sai] 🙏 Sending: 'Thank you for explaining! I understand Python better now.'

[Professor Smith] 😊 Received: 'Thank you for explaining! I understand Python better now.'
[Professor Smith] Great! Happy to help you learn about Python!


[Professor Smith] 📝 Received question from Sai
Topic: Agents
Question: Agents - How do software agents work?

[Professor Smith] 🧠 

In [7]:
print("\n🔹 QUESTION 1: Python Programming\n")

question1 = Question(
    topic="Python",
    question_text="Python - What is Python and why is it popular?",
    student_name="Sai"
)

await runtime.send_message(question1, AgentId("teacher_agent", "default"))
await asyncio.sleep(2)



🔹 QUESTION 1: Python Programming



In [8]:
print("\n🔹 QUESTION 2: Software Agents\n")

question2 = Question(
    topic="Agents",
    question_text="Agents - How do software agents work?",
    student_name="Sai"
)

await runtime.send_message(question2, AgentId("teacher_agent", "default"))
await asyncio.sleep(2)


🔹 QUESTION 2: Software Agents



In [9]:
print("\n🔹 QUESTION 3: Artificial Intelligence\n")

question3 = Question(
    topic="AI",
    question_text="AI - What is Artificial Intelligence?",
    student_name="Sai"
)

await runtime.send_message(question3, AgentId("teacher_agent", "default"))
await asyncio.sleep(2)


🔹 QUESTION 3: Artificial Intelligence



In [10]:
print("\n" + "="*70)
print(" "*20 + "SESSION SUMMARY")
print("="*70)
print(f"Total Questions Asked: 3")
print(f"Topics Covered: Python, Agents, AI")
print(f"Total Messages Exchanged: 9 (3 questions + 3 answers + 3 acknowledgments)")
print("="*70)

# Stop runtime
print("\n[SYSTEM] ⏹️  Stopping runtime...")
await runtime.stop()
print("[SYSTEM] ✅ Learning session complete!")
print("="*70)


                    SESSION SUMMARY
Total Questions Asked: 3
Topics Covered: Python, Agents, AI
Total Messages Exchanged: 9 (3 questions + 3 answers + 3 acknowledgments)

[SYSTEM] ⏹️  Stopping runtime...
[SYSTEM] ✅ Learning session complete!
